# Aplicações com MediaPipe

Os detectores de landmarks do MediaPipe rodam em tempo real na CPU, o que permite usar a webcam como entrada. Com os pontos em cada frame, uma aplicação precisa apenas de geometria: ângulos, distâncias e comparações entre pontos.

Este notebook mostra ao vivo, na webcam, os detectores de pose, de mãos e de rosto, cada um seguido de aplicações sobre os seus pontos: um contador de agachamentos com a pose; um controle do volume do computador e um contador de dedos com as mãos; barras de expressão e um alerta de sonolência com o rosto. O notebook precisa rodar na sua máquina, porque o OpenCV acessa a câmera local e abre uma janela própria para o vídeo.

In [ ]:
import os
import subprocess
import time
import urllib.request

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import BaseOptions, vision
from mediapipe.tasks.python.vision import drawing_styles, drawing_utils

## Detectores

Cada detector é um arquivo `.task`, baixado uma vez para a pasta `data`. O OpenCV lê os frames em BGR e o MediaPipe espera RGB, por isso a `to_mp_image` inverte os canais. Os pontos vêm normalizados entre 0 e 1, e a `to_pixels` os converte em um array em pixels, onde as contas de ângulo e distância não saem distorcidas em imagens que não são quadradas.

In [ ]:
MODELS = "https://storage.googleapis.com/mediapipe-models"


def download(url, name):
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path


def to_mp_image(frame):
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))


def to_pixels(landmarks, frame):
    H, W = frame.shape[:2]
    return np.array([[point.x * W, point.y * H] for point in landmarks])   # [K, 2]


pose_detector = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=download(f"{MODELS}/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task", "pose.task")),
))
hand_detector = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=download(f"{MODELS}/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task", "hand.task")),
    num_hands=2,
))
face_detector = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=download(f"{MODELS}/face_landmarker/face_landmarker/float16/latest/face_landmarker.task", "face.task")),
    output_face_blendshapes=True,
))

## Webcam

A `run_webcam` lê a câmera frame a frame, chama `process(frame)`, que desenha o resultado de cada aplicação sobre o frame, e mostra o frame em uma janela. A tecla `q` ou `Esc` fecha a janela. O frame é espelhado antes de tudo, como em uma chamada de vídeo, para a mão direita aparecer do lado direito da tela.

Os detectores rodam com `detect`, um frame por vez. O MediaPipe oferece também os modos `VIDEO` e `LIVE_STREAM`, que aproveitam os frames anteriores, mas exigem um timestamp crescente a cada chamada e um detector exclusivo por vídeo. Para estes exemplos o ganho de tempo foi pequeno, e ficamos com a forma mais simples.

In [ ]:
def run_webcam(process, source=0):
    camera = cv2.VideoCapture(source)   # 0 é a webcam; também aceita o caminho de um vídeo
    while True:
        ok, frame = camera.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)   # espelha na horizontal
        process(frame)
        cv2.imshow("webcam", frame)
        if cv2.waitKey(1) & 0xFF in (ord("q"), 27):   # q ou Esc
            break
    camera.release()
    cv2.destroyAllWindows()

## Pose

O detector de pose devolve 33 pontos do corpo, cada um com uma `visibility`, a confiança de que o ponto aparece na imagem. O `drawing_utils` do MediaPipe só desenha os pontos com visibilidade alta, então ao se aproximar da câmera as pernas somem do desenho. Afaste-se até o corpo inteiro aparecer.

In [ ]:
def draw_pose(frame, landmarks):
    drawing_utils.draw_landmarks(
        frame, landmarks, vision.PoseLandmarksConnections.POSE_LANDMARKS,
        drawing_styles.get_default_pose_landmarks_style(),
    )


def process_pose(frame):
    for landmarks in pose_detector.detect(to_mp_image(frame)).pose_landmarks:
        draw_pose(frame, landmarks)


run_webcam(process_pose)

### Contador de agachamentos

Um agachamento é visto pelo ângulo do joelho, entre a coxa e a canela. Com os pontos do quadril $q$, do joelho $j$ e do tornozelo $t$, o ângulo no joelho é

$$\theta = \arccos \frac{(q - j) \cdot (t - j)}{\|q - j\|\,\|t - j\|}$$

perto de 180° com a perna esticada e abaixo de 90° no fundo do agachamento. De lado para a câmera, uma das pernas fica escondida atrás da outra, e por isso medimos a perna com maior `visibility`: os pontos 23, 25 e 27 de um lado e 24, 26 e 28 do outro. Se nem ela passa de 0,5, a perna está fora da imagem, os pontos são chutes do modelo, e o frame é ignorado.

Contar os frames com o joelho dobrado contaria cada agachamento dezenas de vezes. O contador guarda um estado, embaixo ou em pé, e soma uma repetição só na volta de embaixo para em pé. Os dois limites, 90° para descer e 160° para subir, ficam afastados de propósito: com um limite só, o ângulo tremendo em volta dele contaria várias repetições em uma.

In [ ]:
LEGS = [(23, 25, 27), (24, 26, 28)]   # quadril, joelho e tornozelo de cada lado
squats = {"count": 0, "down": False}


def process_squats(frame):
    result = pose_detector.detect(to_mp_image(frame))
    if result.pose_landmarks:
        landmarks = result.pose_landmarks[0]
        draw_pose(frame, landmarks)
        points = to_pixels(landmarks, frame)

        # Perna mais visível, e só se ela aparece de fato na imagem
        hip, knee, ankle = max(LEGS, key=lambda leg: sum(landmarks[k].visibility for k in leg))
        if min(landmarks[k].visibility for k in (hip, knee, ankle)) > 0.5:
            u, v = points[hip] - points[knee], points[ankle] - points[knee]
            angle = np.degrees(np.arccos(u @ v / (np.linalg.norm(u) * np.linalg.norm(v))))

            if angle < 90:
                squats["down"] = True
            if angle > 160 and squats["down"]:
                squats["down"] = False
                squats["count"] += 1
            cv2.putText(frame, f"{angle:.0f}", tuple(points[knee].astype(int) + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)

    cv2.putText(frame, str(squats["count"]), (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 200, 255), 6)
    cv2.putText(frame, "embaixo" if squats["down"] else "em pe", (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)


run_webcam(process_squats)

## Mãos

O detector de mãos devolve 21 pontos por mão, até duas mãos: o punho (0) e quatro pontos por dedo, da base até a ponta. O polegar é 1 a 4, o indicador 5 a 8, o médio 9 a 12, o anelar 13 a 16 e o mínimo 17 a 20.

In [ ]:
def draw_hand(frame, landmarks):
    drawing_utils.draw_landmarks(
        frame, landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS,
        drawing_styles.get_default_hand_landmarks_style(), drawing_styles.get_default_hand_connections_style(),
    )


def process_hands(frame):
    for landmarks in hand_detector.detect(to_mp_image(frame)).hand_landmarks:
        draw_hand(frame, landmarks)


run_webcam(process_hands)

### Controle de volume

O volume segue a abertura da pinça entre a ponta do polegar, o ponto 4, e a ponta do indicador, o ponto 8. A distância em pixels não serve sozinha: a mesma mão, mais longe da câmera, aparece menor, e o volume mudaria só porque a pessoa se afastou. Dividimos a pinça pelo tamanho da palma, a distância do punho (0) até a base do dedo médio (9), que encolhe na mesma proporção:

$$\text{pinça} = \frac{\|p_4 - p_8\|}{\|p_0 - p_9\|}$$

A pinça de 0,2, com os dedos quase encostados, vira volume 0, e a de 1,2, com a pinça bem aberta, vira 100. Os pontos tremem alguns pixels de um frame para o outro, mesmo com a mão parada, e o volume tremeria junto. Suavizamos o valor com uma média móvel exponencial, $v \leftarrow 0{,}7\,v + 0{,}3\,v_{\text{novo}}$, que segue a mão com um pequeno atraso.

O volume do sistema muda pelo comando `wpctl`, do PipeWire, o servidor de áudio das distribuições Linux atuais. No macOS, o comando equivalente é `osascript -e "set volume output volume 50"`; no Windows não há comando nativo, e é preciso uma biblioteca como a `pycaw`.

In [ ]:
MIN_PINCH, MAX_PINCH = 0.2, 1.2
volume = {"level": 50.0}   # nível suavizado, de 0 a 100


def set_volume(level):
    subprocess.run(["wpctl", "set-volume", "@DEFAULT_AUDIO_SINK@", f"{level / 100:.2f}"])


def process_volume(frame):
    result = hand_detector.detect(to_mp_image(frame))
    if result.hand_landmarks:
        draw_hand(frame, result.hand_landmarks[0])
        points = to_pixels(result.hand_landmarks[0], frame)
        cv2.line(frame, tuple(points[4].astype(int)), tuple(points[8].astype(int)), (0, 220, 0), 3)

        pinch = np.linalg.norm(points[4] - points[8]) / np.linalg.norm(points[0] - points[9])
        target = 100 * np.clip((pinch - MIN_PINCH) / (MAX_PINCH - MIN_PINCH), 0, 1)
        volume["level"] = 0.7 * volume["level"] + 0.3 * target
        set_volume(round(volume["level"]))

    # Barra vertical com o volume
    level = round(volume["level"])
    cv2.rectangle(frame, (20, 100), (60, 400), (255, 255, 255), 2)
    cv2.rectangle(frame, (20, 400 - 3 * level), (60, 400), (0, 200, 255), -1)
    cv2.putText(frame, f"{level}%", (15, 440), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)


run_webcam(process_volume)

Os limites `MIN_PINCH` e `MAX_PINCH` dependem da mão de cada pessoa e de como ela faz a pinça. Se o volume não chega a 0 ou a 100, ajuste os dois valores. Com a mão de perfil para a câmera, o polegar e o indicador ficam um atrás do outro na imagem, a pinça medida encolhe e o volume cai, mesmo com os dedos abertos.

### Números com a mão

Para reconhecer o número feito com os dedos, de 0 a 10, contamos os dedos esticados de cada mão e somamos. A regra mais óbvia compara alturas: com a mão em pé, um dedo esticado tem a ponta acima da articulação dois pontos abaixo. Ela erra com a mão deitada ou de cabeça para baixo, porque a altura depende de como a mão está girada na imagem, e o gesto não depende.

Distâncias entre pontos não mudam quando a mão gira. Um dedo esticado tem a ponta mais longe do punho que a articulação dois pontos abaixo, e um dedo dobrado traz a ponta de volta para a palma, mais perto do punho:

$$\text{dedo esticado} \iff \|p_{\text{ponta}} - p_0\| > \|p_{\text{ponta} - 2} - p_0\|$$

O polegar dobra para o lado, por cima da palma, e a ponta dele continua longe do punho. Para ele comparamos as distâncias até a base do mínimo, o ponto 17: esticado, a ponta fica mais longe do ponto 17 que a base do polegar, o ponto 2; dobrado sobre a palma, fica mais perto.

A alternativa seria treinar um classificador sobre os pontos, com fotos de cada número. Aqui ele não é necessário, porque o número é a quantidade de dedos esticados, e a regra não precisa de dados.

In [ ]:
def fingers_up(points):
    d = lambda i, j: np.linalg.norm(points[i] - points[j])   # distância entre os pontos i e j
    thumb = d(4, 17) > d(2, 17)
    others = [d(tip, 0) > d(tip - 2, 0) for tip in [8, 12, 16, 20]]
    return [thumb] + others   # 5 bools, do polegar ao mínimo


def process_numbers(frame):
    total = 0
    for landmarks in hand_detector.detect(to_mp_image(frame)).hand_landmarks:
        draw_hand(frame, landmarks)
        total += sum(fingers_up(to_pixels(landmarks, frame)))
    cv2.putText(frame, str(total), (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 200, 255), 6)


run_webcam(process_numbers)

## Rosto

O detector de rosto devolve 478 pontos. Desenhamos os pontos e a malha de triângulos que os liga, a mesma usada para aplicar maquiagem e máscaras virtuais em aplicativos de vídeo.

In [ ]:
def draw_face(frame, landmarks):
    drawing_utils.draw_landmarks(
        frame, landmarks, vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
        drawing_utils.DrawingSpec(color=(0, 0, 255), thickness=1, circle_radius=1),   # pontos
        drawing_styles.get_default_face_mesh_tesselation_style(),                      # malha
    )


def process_face(frame):
    for landmarks in face_detector.detect(to_mp_image(frame)).face_landmarks:
        draw_face(frame, landmarks)


run_webcam(process_face)

### Barras de expressão

Com `output_face_blendshapes=True`, o detector devolve também 52 blendshapes, escores entre 0 e 1 para expressões como sorrir, piscar ou abrir a boca. Mostramos cinco deles em barras. Sorria, pisque um olho de cada vez e abra a boca para ver as barras mudarem.

In [ ]:
BLENDSHAPES = ["mouthSmileLeft", "mouthSmileRight", "eyeBlinkLeft", "eyeBlinkRight", "jawOpen"]


def process_expressions(frame):
    result = face_detector.detect(to_mp_image(frame))
    if not result.face_landmarks:
        return
    draw_face(frame, result.face_landmarks[0])

    # scores é um dict com o escore de cada blendshape, ex.: {"jawOpen": 0.03, ...}
    scores = {category.category_name: category.score for category in result.face_blendshapes[0]}
    for k, name in enumerate(BLENDSHAPES):
        y = 30 + 30 * k
        cv2.rectangle(frame, (10, y - 15), (10 + int(150 * scores[name]), y), (0, 200, 255), -1)
        cv2.putText(frame, name, (170, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)


run_webcam(process_expressions)

### Alerta de sonolência

Uma piscada fecha os olhos por uma fração de segundo; um motorista cochilando os mantém fechados por mais tempo. O alerta usa os blendshapes `eyeBlinkLeft` e `eyeBlinkRight`: os olhos contam como fechados quando os dois passam de 0,5, e a tela fica vermelha se continuarem fechados por mais de 1 segundo.

Um único frame com os olhos fechados não diz nada, porque pode ser uma piscada. Por isso o alerta guarda o instante em que os olhos se fecharam e mede o tempo desde então.

In [ ]:
BLINK_THRESHOLD, MAX_CLOSED_SECONDS = 0.5, 1.0
eyes = {"closed_since": None}   # instante em que os olhos se fecharam, ou None se estão abertos


def process_drowsiness(frame):
    result = face_detector.detect(to_mp_image(frame))
    if not result.face_landmarks:
        return
    draw_face(frame, result.face_landmarks[0])

    scores = {category.category_name: category.score for category in result.face_blendshapes[0]}
    closed = scores["eyeBlinkLeft"] > BLINK_THRESHOLD and scores["eyeBlinkRight"] > BLINK_THRESHOLD
    if not closed:
        eyes["closed_since"] = None
    elif eyes["closed_since"] is None:
        eyes["closed_since"] = time.time()

    seconds = 0 if eyes["closed_since"] is None else time.time() - eyes["closed_since"]
    cv2.putText(frame, f"olhos fechados: {seconds:.1f} s", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
    if seconds > MAX_CLOSED_SECONDS:
        H, W = frame.shape[:2]
        cv2.rectangle(frame, (0, 0), (W - 1, H - 1), (0, 0, 255), 20)
        cv2.putText(frame, "ACORDE", (W // 2 - 120, H // 2), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 5)


run_webcam(process_drowsiness)

Os blendshapes de piscar também sobem quando a pessoa olha para baixo, porque as pálpebras descem. Olhar o celular por mais de 1 segundo dispara o alerta. Um detector usado de verdade combinaria os olhos com a inclinação da cabeça, que vem da matriz de transformação do rosto, ativada com `output_facial_transformation_matrixes=True`.